# VT-GraF Benchmark: Visible--Thermal Granular Faults under severe clutter
## Comparison: Unsupervised Generalized Frangi Graph vs. Standard Frangi + SAM 2

This notebook reproduces the comparison on the **VT-GraF-Dataset** (multimodal asphalt crack dataset with severe granular clutter) between:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised multimodal graph-based method (using MST + Betweenness Centrality on GPU).
2. **Baseline (Standard Frangi on Thermal + SAM 2 on Visible)**: Zero-shot SAM 2 prompted automatically by standard Frangi filter responses from the thermal modality.

---
### Technical Setup:
- Automatically downloads the **VT-GraF-Dataset** from Google Drive.
- Installs **Segment Anything 2 (SAM 2)** from Meta's repository and downloads the `sam2_hiera_large.pt` weights.
- Imports the custom GPU graph extraction package.
- Runs both methods and outputs comparative metrics: Jaccard (IoU), Tversky index, and Wasserstein distance.


## 1. Environment Setup & Datasets
We install the required libraries (including Meta's SAM 2 repository) and retrieve the pre-trained weights.


In [ ]:
# Install required packages
!pip install -q gdown POT scikit-image opencv-python matplotlib scipy torch torchvision

# Install SAM 2 from Meta's source
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Download SAM 2 Large weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt


### Download the VT-GraF-Dataset
If the dataset is not already present, we download it from Google Drive using `gdown`.


In [ ]:
import os
import gdown
from pathlib import Path

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'VT-GraF-Dataset'

def check_dataset_exists():
    for path in Path('.').rglob('Fissure 1'):
        return True
    return False

if not check_dataset_exists():
    print("Downloading VT-GraF-Dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download completed.")
else:
    print("Dataset already present.")


## 2. Load the Generalized Frangi Graph Modules
We append the package path (handling both local repository execution and Google Colab environments) and load the required modules.


In [ ]:
import sys
import os
import subprocess
from pathlib import Path

def find_isprs_src():
    # 1. Check current directory
    if Path('ISPRS/src').exists():
        return Path('ISPRS').resolve()
    # 2. Check under /content/drive/MyDrive (if mounted in Colab)
    try:
        for p in Path('/content/drive/MyDrive').rglob('ISPRS/src'):
            return p.parent.resolve()
    except Exception:
        pass
    # 3. Check under current directory recursively (e.g. if cloned elsewhere)
    for p in Path('.').rglob('ISPRS/src'):
        if 'Generalized-Frangi' in str(p):
            return p.parent.resolve()
    return None

isprs_path = find_isprs_src()

if isprs_path is None:
    print("Running in Colab: Attempting to clone repository to access code modules...")
    try:
        # Run git clone with a timeout of 30 seconds to avoid hanging forever
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "https://github.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset.git"
        ], check=True, timeout=30)
        
        subprocess.run([
            "git", "-C", "Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset",
            "sparse-checkout", "set", "ISPRS"
        ], check=True)
        
        isprs_path = Path("Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS").resolve()
        print("Repository cloned successfully!")
    except Exception as e:
        print("\n" + "!"*60)
        print("ERROR: Failed to clone repository from GitHub (network limit or private repository).")
        print("!"*60)
        print("FALLBACK OPTIONS:")
        print("1. MOUNT GOOGLE DRIVE:")
        print("   If you have the repository on your Google Drive, run the following code in a new cell:")
        print("      from google.colab import drive")
        print("      drive.mount('/content/drive')")
        print("   Then re-run this cell; it will automatically find the files in your Drive.")
        print("\n2. UPLOAD DIRECTLY:")
        print("   Zip the 'ISPRS' folder locally, upload it to the Colab files tab (left sidebar),")
        print("   unzip it in Colab using: !unzip ISPRS.zip")
        print("   Then re-run this cell.")
        print("!"*60 + "\n")
        
if isprs_path is not None:
    sys.path.insert(0, str(isprs_path))
    print(f"Added code modules path: {isprs_path}")
    from src import (
        VTGraFDataset, extract_frangi_graph_gpu, skeletonize_lee,
        thicken, compute_metrics, wasserstein_distance_skeletons
    )
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    dataset = VTGraFDataset('.')
else:
    raise RuntimeError("Could not find or load the 'ISPRS/src' modules. Please follow one of the fallback options above.")


## 3. Initialize SAM 2 Predictor
We load the pre-trained SAM 2 Large model on GPU (or CPU as fallback) and initialize the image predictor.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, sam2_checkpoint, device=device))
print("SAM 2 Large predictor loaded successfully!")


## 4. Define SAM 2 Baseline Prompter
We define the baseline approach which extracts points from standard thermal Frangi responses and queries SAM 2 on the high-resolution visible image.


In [ ]:
import numpy as np
import cv2
from skimage.filters import frangi

def run_baseline_frangi_sam2(sample, predictor, sigmas=[10, 20, 30, 40], num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    # Note: We use sigmas to match our method's scales exactly
    frangi_response = frangi(img_ir_np / 255.0, sigmas=sigmas)
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts,
        'frangi_response': frangi_response,
        'binary_mask': binary_mask,
        'skel': skel
    }


## 5. Quantitative Evaluation & Benchmark
We evaluate both methods on all 5 images of the VT-GraF-Dataset.
- For **Ours (Generalized Frangi Graph)**, we use parameters: `K=2` (dual triangle graph), scale set `Σ=[20,30,40]` and weights `visible: 1/3, infrared: 2/3`. Skeletons are thickened to **5 pixels** for final evaluation.
- For the **Baseline (Frangi Thermal + SAM 2)**, we generate the mask, skeletonize it, and thicken it to **5 pixels** similarly.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import torch

def compute_scale_specific_hessian(image_tensor, sigma, device):
    from src.frangi_hessian import FrangiHessianGPU
    fh = FrangiHessianGPU([sigma], device=device)
    ixx, ixy, iyy = fh.compute_hessian(image_tensor, sigma)
    trace = ixx + iyy
    disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
    spectral_norm_local = (torch.abs(trace) + disc) / 2.0
    max_norm = torch.max(spectral_norm_local) + 1e-8
    
    ixx_n = ixx / max_norm
    iyy_n = iyy / max_norm
    ixy_n = ixy / max_norm
    
    λ1, λ2, θ = fh.compute_eigenvalues_and_vectors(ixx_n, ixy_n, iyy_n)
    S_norm = torch.zeros_like(λ2)
    mask_pos = λ2 > 0
    S_norm[mask_pos] = λ2[mask_pos]
    return S_norm.cpu().numpy()

def compute_fused_scale_specific_hessian(visible_tensor, ir_tensor, weights, sigma, device):
    from src.frangi_hessian import FrangiHessianGPU
    fh = FrangiHessianGPU([sigma], device=device)
    
    fused_ixx = None
    for mod, w in weights.items():
        if w > 0:
            img = visible_tensor if mod == 'visible' else ir_tensor
            ixx, ixy, iyy = fh.compute_hessian(img, sigma)
            trace = ixx + iyy
            disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
            spectral_norm_local = (torch.abs(trace) + disc) / 2.0
            max_norm = torch.max(spectral_norm_local) + 1e-8
            
            if fused_ixx is None:
                fused_ixx = w * (ixx / max_norm)
                fused_ixy = w * (ixy / max_norm)
                fused_iyy = w * (iyy / max_norm)
            else:
                fused_ixx += w * (ixx / max_norm)
                fused_ixy += w * (ixy / max_norm)
                fused_iyy += w * (iyy / max_norm)
                
    λ1, λ2, θ = fh.compute_eigenvalues_and_vectors(fused_ixx, fused_ixy, fused_iyy)
    S_norm = torch.zeros_like(λ2)
    mask_pos = λ2 > 0
    S_norm[mask_pos] = λ2[mask_pos]
    return S_norm.cpu().numpy()

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}
sigmes_benchmark = [10, 20, 30, 40]

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[10,20,30,40]
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    
    # Get Hessian (max over scales of λ2) of each modality individually
    max_S_vis, _, _, _, _ = extract_frangi_graph_gpu(
        imgs_i, {'visible': 1.0, 'infrared': 0.0}, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    max_S_ir, _, _, _, _ = extract_frangi_graph_gpu(
        imgs_i, {'visible': 0.0, 'infrared': 1.0}, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    
    # Compute scale-specific Hessians (for 10, 20, 30, 40)
    scale_visualizations = {}
    for sigma in sigmes_benchmark:
        vis_sig = compute_scale_specific_hessian(imgs_i['visible'], float(sigma), device)
        ir_sig = compute_scale_specific_hessian(imgs_i['infrared'], float(sigma), device)
        fused_sig = compute_fused_scale_specific_hessian(imgs_i['visible'], imgs_i['infrared'], weights_ours, float(sigma), device)
        scale_visualizations[int(sigma)] = {
            'visible': vis_sig,
            'infrared': ir_sig,
            'fused': fused_sig
        }
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # --- 2. RUN BASELINE (Frangi Thermal + SAM 2) ---
    baseline_outputs = run_baseline_frangi_sam2(sample, predictor, sigmas=sigmes_benchmark, num_prompts=12)
    pred_baseline = baseline_outputs['pred_mask']
    sampled_pts = baseline_outputs['pts']
    sk_pred_baseline = skeletonize_lee(pred_baseline)
    sk_pred_thick_baseline = thicken(sk_pred_baseline, pixels=5)
    
    # --- 3. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline
    jac_base, tv_base = compute_metrics(sk_pred_thick_baseline, sk_gt_thick)
    wass_base = wasserstein_distance_skeletons(sk_pred_thick_baseline, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'Baseline_Jaccard': jac_base,
        'Baseline_Tversky': tv_base,
        'Baseline_Wasserstein': wass_base
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours_pred': pred_ours,
        'ours': sk_pred_thick_ours,
        'ours_centrality': centrality_i,
        'ours_fused_frangi': max_S_global,
        'ours_vis_frangi': max_S_vis,
        'ours_ir_frangi': max_S_ir,
        'ours_similarity': sim_img,
        'ours_tau_mask': diagnostics_i.get('tau_mask', np.zeros_like(pred_ours)),
        'ours_comp_mask': diagnostics_i.get('comp_mask', np.zeros_like(pred_ours)),
        'baseline': sk_pred_thick_baseline,
        'baseline_mask': pred_baseline,
        'points': sampled_pts,
        'baseline_frangi': baseline_outputs['frangi_response'],
        'baseline_binary': baseline_outputs['binary_mask'],
        'baseline_skel': baseline_outputs['skel'],
        'scale_visualizations': scale_visualizations
    }

df_res = pd.DataFrame(results)


## 6. Comparison Results Table
We display the quantitative results and compute the mean scores across all fissures.


In [ ]:
# Format and display the results
pd.set_option('display.precision', 4)
display(df_res)

# Print Global Averages
print("\n" + "="*50)
print("--- SUMMARY STATISTICS ---")
print("="*50)
print(f"Jaccard (IoU) Mean   : Ours = {df_res['Ours_Jaccard'].mean():.4f} | Baseline = {df_res['Baseline_Jaccard'].mean():.4f}")
print(f"Tversky Index Mean   : Ours = {df_res['Ours_Tversky'].mean():.4f} | Baseline = {df_res['Baseline_Tversky'].mean():.4f}")
print(f"Wasserstein Dist Mean: Ours = {df_res['Ours_Wasserstein'].mean():.4f} | Baseline = {df_res['Baseline_Wasserstein'].mean():.4f}")


## 7. Visual Inspection
We plot each processing step side-by-side to visually inspect the qualitative output.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Create directory to save generated comparison images
os.makedirs('results_images', exist_ok=True)

for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    h, w = vis_data['visible'].shape
    
    # --- 1. 3x3 COMPARISON GRID ---
    fig, axes = plt.subplots(3, 3, figsize=(18, 18))
    
    # ROW 1: Raw Inputs & Ground Truth
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    
    # ROW 2: Hessians (max over scales of λ2)
    axes[1, 0].imshow(vis_data['ours_vis_frangi'], cmap='magma')
    axes[1, 1].imshow(vis_data['ours_ir_frangi'], cmap='magma')
    axes[1, 2].imshow(vis_data['ours_fused_frangi'], cmap='magma')
    
    # ROW 3: Outputs & Overlays
    axes[2, 0].imshow(vis_data['ours'], cmap='hot')
    axes[2, 1].imshow(vis_data['baseline'], cmap='hot')
    
    # Overlay: White for GT, Green for Ours, Red for SAM 2
    rgba_gt = np.zeros((h, w, 4))
    rgba_gt[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4]
    rgba_ours = np.zeros((h, w, 4))
    rgba_ours[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4]
    rgba_base = np.zeros((h, w, 4))
    rgba_base[vis_data['baseline'] > 0] = [1.0, 0.0, 0.0, 0.4]
    
    axes[2, 2].imshow(vis_data['visible'], cmap='gray')
    axes[2, 2].imshow(rgba_gt)
    axes[2, 2].imshow(rgba_ours)
    axes[2, 2].imshow(rgba_base)
    
    # Disable axes and titles
    for ax in axes.flat:
        ax.axis('off')
        
    plt.tight_layout()
    
    # Save the clean grid without titles/axes
    grid_filename = f"results_images/comparison_grid_{name.replace(' ', '_')}.png"
    plt.savefig(grid_filename, bbox_inches='tight', pad_inches=0)
    plt.show()
    print(f"Saved comparison grid for {name} to {grid_filename}")
    
    # --- 2. 4x3 SCALE-SPECIFIC HESSIAN GRID ---
    fig_s, axes_s = plt.subplots(4, 3, figsize=(18, 24))
    
    for r_idx, sigma in enumerate([10, 20, 30, 40]):
        scale_data = vis_data['scale_visualizations'][sigma]
        
        # Col 0: Visible scale-specific Hessian
        axes_s[r_idx, 0].imshow(scale_data['visible'], cmap='magma')
        
        # Col 1: Infrared scale-specific Hessian
        axes_s[r_idx, 1].imshow(scale_data['infrared'], cmap='magma')
        
        # Col 2: Fused scale-specific Hessian
        axes_s[r_idx, 2].imshow(scale_data['fused'], cmap='magma')
        
    # Disable axes and titles
    for ax in axes_s.flat:
        ax.axis('off')
        
    plt.tight_layout()
    
    # Save the scale-specific grid without titles/axes
    scale_filename = f"results_images/scales_grid_{name.replace(' ', '_')}.png"
    plt.savefig(scale_filename, bbox_inches='tight', pad_inches=0)
    plt.show()
    print(f"Saved scale-specific grid for {name} to {scale_filename}")
